# 03 - Validation: where do the models fail?

Both models on their own validation frames, before anyone looks at the test videos:

1. Dice per class for model A (random validation frames) and model B (validation videos).
2. Which classes model B confuses with which.
3. A first look at the warning question on model B's validation frames. This is exploratory: the test that counts is in notebook 04, with the rule fixed in `docs/plan.md`.

Kaggle: Accelerator GPU T4, Internet on, the dataset `newslab/cholecseg8k` and the output of notebook 02 as inputs.
Output: `out/val_per_class.csv`, `out/val_frames_B.csv`, `out/val_summary.json`, `out/confusion_B.png`, `out/worst_val_frames_B.png`, zipped as `results_03.zip`.

In [ ]:
import glob, json, math, os, re, shutil, time
from pathlib import Path
from multiprocessing import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

KAGGLE = Path('/kaggle').exists()
WORK = Path('/kaggle/working') if KAGGLE else Path('.')
DATA = Path(os.environ.get('DATA_ROOT', '/kaggle/input'))     # where the CholecSeg8k frames are
INPUT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))   # where the output of the previous notebook is
OUT = WORK / 'out'
OUT.mkdir(exist_ok=True)
H, W = 256, 448   # frames are 480 x 854; resized to a size the network can halve five times

CLASSES = ['black background', 'abdominal wall', 'liver', 'gastrointestinal tract', 'fat', 'grasper',
           'connective tissue', 'blood', 'cystic duct', 'L-hook electrocautery', 'gallbladder',
           'hepatic vein', 'liver ligament']
N_CLASSES = len(CLASSES)
IGNORE = 255      # mask pixels with a value that belongs to no class


def find_input(name):
    """Find a file from the previous notebook's output under INPUT."""
    hits = sorted(glob.glob(str(INPUT / '**' / name), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {INPUT}: add the output of the previous notebook as input')
    return Path(hits[0])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PRETRAINED = os.environ.get('PRETRAINED', '1') == '1'   # ImageNet weights for the encoder (needs Internet on)
torch.manual_seed(0); np.random.seed(0)
print('device:', DEVICE)

split = pd.read_csv(find_input('split.csv'))
mapping = {int(k): v for k, v in json.loads(find_input('mapping.json').read_text()).items()}
classes_info = json.loads(find_input('classes.json').read_text())
LUT = np.full(256, IGNORE, np.uint8)            # mask value -> class id
for value, cls in mapping.items():
    LUT[value] = cls
print(split.groupby('split_B').video.nunique().to_dict(), 'videos per part (video split)')

In [ ]:
def load_pair(path):
    """One frame and its class mask, both resized to H x W."""
    img = Image.open(path).convert('RGB').resize((W, H), Image.BILINEAR)
    m = np.array(Image.open(path.replace('_endo.png', '_endo_watershed_mask.png')))
    if m.ndim == 3:
        m = m[..., 0]                               # the three channels hold the same value
    m = Image.fromarray(LUT[m]).resize((W, H), Image.NEAREST)
    return np.asarray(img, np.uint8), np.asarray(m, np.uint8)


def load_frames(rows):
    """Read the frames of a table (one row per frame) into two arrays, in table order."""
    t0 = time.time()
    with Pool(os.cpu_count()) as pool:
        pairs = pool.map(load_pair, rows.path.tolist(), chunksize=16)
    imgs = np.stack([p[0] for p in pairs])
    masks = np.stack([p[1] for p in pairs])
    print(f'{len(rows)} frames loaded in {time.time() - t0:.0f} s, {imgs.nbytes / 1e9:.1f} GB')
    return imgs, masks

In [ ]:
class DoubleConv(nn.Module):
    """Conv 3x3 -> BatchNorm -> ReLU, two times. Height and width stay the same."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    """Double the size, join the encoder output of the same size (skip connection), DoubleConv."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class UNetResNet34(nn.Module):
    """U-Net whose encoder is a ResNet-34 trained on ImageNet; the decoder is trained from scratch."""
    def __init__(self, n_classes=13, pretrained=True):
        super().__init__()
        r = torchvision.models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)   # 1/2 size, 64 channels
        self.pool = r.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.up4 = Up(512, 256, 256)                        # 1/32 -> 1/16
        self.up3 = Up(256, 128, 128)                        # 1/16 -> 1/8
        self.up2 = Up(128, 64, 64)                          # 1/8 -> 1/4
        self.up1 = Up(64, 64, 64)                           # 1/4 -> 1/2
        self.head = nn.Sequential(DoubleConv(64, 32), nn.Conv2d(32, n_classes, 1))

    def forward(self, x):
        s0 = self.stem(x)                     # 1/2
        s1 = self.layer1(self.pool(s0))       # 1/4, 64
        s2 = self.layer2(s1)                  # 1/8, 128
        s3 = self.layer3(s2)                  # 1/16, 256
        s4 = self.layer4(s3)                  # 1/32, 512
        x = self.up1(self.up2(self.up3(self.up4(s4, s3), s2), s1), s0)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.head(x)                   # 13 logits per pixel


MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_input(x_uint8):
    """(B, H, W, 3) uint8 -> (B, 3, H, W) float, normalised like the ImageNet training data."""
    x = x_uint8.to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    return (x - MEAN) / STD

In [ ]:
@torch.no_grad()
def evaluate(model, imgs, masks, batch=32):
    """Run the model on frames and score it.

    Returns the confusion matrix over all frames (rows: true class, columns: predicted class) and
    one row per frame: mean Dice over the classes in that frame, and two uncertainty numbers.
    Pixels whose mask value belongs to no class are left out.
    """
    model.eval()
    total = torch.zeros(N_CLASSES * N_CLASSES, dtype=torch.long, device=DEVICE)
    rows = []
    for k in range(0, len(imgs), batch):
        x = to_input(torch.from_numpy(imgs[k:k + batch]))
        y = torch.from_numpy(masks[k:k + batch]).to(DEVICE).long()
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            logits = model(x)
        prob = logits.float().softmax(1)
        pred = prob.argmax(1)
        entropy = -(prob * prob.clamp_min(1e-8).log()).sum(1)        # per pixel, 0 = certain
        for i in range(len(y)):
            valid = y[i] != IGNORE
            cm = torch.bincount(y[i][valid] * N_CLASSES + pred[i][valid], minlength=N_CLASSES ** 2)
            total += cm
            cm = cm.view(N_CLASSES, N_CLASSES).cpu().numpy()
            inter, true_px, pred_px = np.diag(cm), cm.sum(1), cm.sum(0)
            seen = (true_px + pred_px)[1:] > 0                         # classes 1-12 in truth or prediction
            dice = (2 * inter / np.maximum(true_px + pred_px, 1))[1:][seen]
            fg = pred[i] != 0                                          # everything except the black border
            rows.append(dict(frame_dice=float(dice.mean()) if len(dice) else np.nan,
                             entropy_fg=float(entropy[i][fg].mean()) if fg.any() else float(entropy[i].mean()),
                             entropy_all=float(entropy[i].mean())))
    return total.view(N_CLASSES, N_CLASSES).cpu().numpy(), pd.DataFrame(rows)


def dice_per_class(cm):
    """Dice per class over all frames; NaN for a class that is not in the ground truth."""
    inter, true_px, pred_px = np.diag(cm), cm.sum(1), cm.sum(0)
    return np.where(true_px > 0, 2 * inter / np.maximum(true_px + pred_px, 1), np.nan)


def mean_dice(cm):
    """Mean Dice over classes 1-12 that are in the ground truth (class 0 is the black border)."""
    return float(np.nanmean(dice_per_class(cm)[1:]))

In [ ]:
def load_model(name):
    model = UNetResNet34(N_CLASSES, pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(find_input(f'model_{name}.pt'), map_location=DEVICE))
    return model.eval()


rows_A = split[split.split_A == 'val'].reset_index(drop=True)
rows_B = split[split.split_B == 'val'].reset_index(drop=True)
imgs_A, masks_A = load_frames(rows_A)
cm_A, frames_A = evaluate(load_model('A'), imgs_A, masks_A)
del imgs_A, masks_A
imgs_B, masks_B = load_frames(rows_B)
cm_B, frames_B = evaluate(load_model('B'), imgs_B, masks_B)
frames_B = pd.concat([rows_B[['video', 'clip_name', 'frame']], frames_B], axis=1)
frames_B.to_csv(OUT / 'val_frames_B.csv', index=False)

logged = {n: float(pd.read_csv(find_input(f'train_log_{n}.csv')).val_mean_dice.max()) for n in 'AB'}
print(f'model A, random validation frames: mean Dice {mean_dice(cm_A):.3f} (logged in 02: {logged["A"]:.3f})')
print(f'model B, validation videos:       mean Dice {mean_dice(cm_B):.3f} (logged in 02: {logged["B"]:.3f})')

## Dice per class

In [ ]:
share = pd.read_csv(find_input('class_pixels.csv')).share_without_border.values
per_class = pd.DataFrame({'class': CLASSES, 'pixel_share': share,
                          'dice_A_val': dice_per_class(cm_A), 'dice_B_val': dice_per_class(cm_B)})
per_class['size'] = ['large' if i in classes_info['large'] else 'small' if i in classes_info['small'] else ''
                     for i in range(N_CLASSES)]
per_class.to_csv(OUT / 'val_per_class.csv', index=False)
print(per_class.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

## Confusion of model B

Row: true class, column: predicted class, each row adds up to 1.

In [ ]:
norm = cm_B / np.maximum(cm_B.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES, rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if norm[i, j] >= 0.05:
            ax.text(j, i, f'{norm[i, j]:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if norm[i, j] > 0.5 else 'black')
ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.tight_layout(); plt.savefig(OUT / 'confusion_B.png', dpi=110); plt.show()

## Warning signal, first look (exploratory)

Same measure as planned for the test: the mean pixel entropy over the area the model calls tissue or instrument (everything except the black border). Do the 10 % most uncertain frames contain the 10 % worst frames? The whole-image entropy is shown next to it.

In [ ]:
from scipy.stats import hypergeom, spearmanr


def warning_test(df, measure):
    df = df.dropna(subset=['frame_dice'])
    n = len(df); k = math.ceil(0.1 * n)
    worst = set(df.nsmallest(k, 'frame_dice').index)
    flagged = set(df.nlargest(k, measure).index)
    hit = len(worst & flagged)
    return dict(measure=measure, frames=n, k=k, caught=hit, expected_by_chance=k * k / n,
                p_value=float(hypergeom.sf(hit - 1, n, k, k)),
                spearman=float(spearmanr(df[measure], df.frame_dice).correlation))


val_warning = [warning_test(frames_B, m) for m in ['entropy_fg', 'entropy_all']]
print(pd.DataFrame(val_warning).to_string(index=False, float_format=lambda x: f'{x:.4g}'))
print()
k = val_warning[0]['k']
print('worst validation frames per video:', frames_B.nsmallest(k, 'frame_dice').video.value_counts().to_dict())

In [ ]:
cmap = plt.get_cmap('tab20', N_CLASSES)
model_B = load_model('B')
worst = frames_B.nsmallest(4, 'frame_dice').index
with torch.no_grad():
    pred = model_B(to_input(torch.from_numpy(imgs_B[worst]))).argmax(1).cpu().numpy()
fig, ax = plt.subplots(3, 4, figsize=(16, 7))
for j, i in enumerate(worst):
    truth = masks_B[i].astype(float); truth[truth == IGNORE] = np.nan
    ax[0, j].imshow(imgs_B[i]); ax[0, j].set_title(f'{frames_B.clip_name[i]} #{frames_B.frame[i]}, Dice {frames_B.frame_dice[i]:.2f}', fontsize=8)
    ax[1, j].imshow(truth, cmap=cmap, vmin=-0.5, vmax=N_CLASSES - 0.5, interpolation='nearest')
    ax[2, j].imshow(pred[j], cmap=cmap, vmin=-0.5, vmax=N_CLASSES - 0.5, interpolation='nearest')
for a in ax.ravel():
    a.axis('off')
ax[1, 0].set_title('truth', loc='left', fontsize=8); ax[2, 0].set_title('model B', loc='left', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'worst_val_frames_B.png', dpi=100); plt.show()

summary = dict(val_mean_dice_A=mean_dice(cm_A), val_mean_dice_B=mean_dice(cm_B), logged=logged,
               frames_A=len(rows_A), frames_B=len(rows_B), warning_B_val=val_warning)
_ = (OUT / 'val_summary.json').write_text(json.dumps(summary, indent=1))
shutil.make_archive(str(WORK / 'results_03'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))